# Study 04 - CasADi MPC Transition

This is the same double-integrator MPC problem from Study 03, reformulated in CasADi. CasADi is a formulation tool here; the control idea is still receding-horizon MPC.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)

import os

import casadi as ca
from scipy.linalg import solve_discrete_are

STUDY_DIR = Path('studies/study_04_casadi_mpc_transition')
OUTPUT_DIR = STUDY_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Same Model, Same Cost, Different Formulation Tool


In [ ]:
dt = 0.1
A = np.array([[1.0, dt], [0.0, 1.0]])
B = np.array([[0.5 * dt**2], [dt]])
Q = np.diag([10.0, 1.0])
R = np.array([[0.2]])
P = solve_discrete_are(A, B, Q, R)
N = 12
u_min, u_max = -1.0, 1.0
x_min = np.array([-2.0, -3.0])
x_max = np.array([2.0, 3.0])
# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.


## Visible CasADi Formulation

States and inputs are decision variables. Dynamics, bounds, stage cost, and terminal cost are still written explicitly.


In [ ]:
def solve_casadi_mpc(x0):
    nx, nu = B.shape
    opti = ca.Opti()
    X = opti.variable(nx, N + 1)
    U = opti.variable(nu, N)
    cost = 0

    opti.subject_to(X[:, 0] == x0)
    for k in range(N):
        xk = X[:, k]
        uk = U[:, k]
        cost += ca.mtimes([xk.T, Q, xk]) + ca.mtimes([uk.T, R, uk])
        opti.subject_to(X[:, k + 1] == ca.mtimes(A, xk) + ca.mtimes(B, uk))
        opti.subject_to(uk >= u_min)
        opti.subject_to(uk <= u_max)
        opti.subject_to(X[:, k] >= x_min)
        opti.subject_to(X[:, k] <= x_max)
    terminal = X[:, N]
    cost += ca.mtimes([terminal.T, P, terminal])
    opti.subject_to(terminal >= x_min)
    opti.subject_to(terminal <= x_max)

    opti.minimize(cost)
    opti.solver('ipopt', {'print_time': False, 'ipopt': {'print_level': 0, 'sb': 'yes'}})
    try:
        sol = opti.solve()
        return float(sol.value(U[0, 0])), str(opti.stats().get('return_status', 'Solve_Succeeded'))
    except RuntimeError:
        return 0.0, str(opti.stats().get('return_status', 'Solve_Failed'))


## Receding-Horizon Simulation


In [ ]:
x0 = np.array([1.5, 0.0])
steps = int(os.environ.get("THIMPC_STUDY04_STEPS", "12"))
X = np.zeros((steps + 1, 2))
U = np.zeros(steps)
statuses = []
X[0] = x0
for k in range(steps):
    u, status = solve_casadi_mpc(X[k])
    U[k] = u
    statuses.append(status)
    X[k + 1] = A @ X[k] + B[:, 0] * u

print('status counts:', {s: statuses.count(s) for s in sorted(set(statuses))})
print('first applied input:', U[0])
print('final state:', X[-1])


In [ ]:
t = np.arange(steps + 1) * dt
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].plot(t, X[:, 0])
axes[1].plot(t, X[:, 1])
axes[2].step(t[:-1], U, where='post')
axes[0].axhline(x_min[0], color='k', linestyle='--', linewidth=0.8)
axes[0].axhline(x_max[0], color='k', linestyle='--', linewidth=0.8)
axes[2].axhline(u_min, color='k', linestyle='--', linewidth=0.8)
axes[2].axhline(u_max, color='k', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('position')
axes[1].set_ylabel('velocity')
axes[2].set_ylabel('input')
axes[2].set_xlabel('time [s]')
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'casadi_mpc_transition.png', dpi=150)
plt.show()


## Student Questions

- Which lines are the same MPC ingredients as Study 03?
- Which lines are CasADi-specific syntax?
- Why does changing tools not change the receding-horizon idea?
